In [21]:
import pandas as pd
import requests
import time
import os
from dotenv import load_dotenv



In [22]:

# ==========================================
# 1. CONFIGURATION
# ==========================================
API_KEY = os.getenv("OPENWEATHER_API_KEY")
INPUT_FILE = "booking_final_gps.csv"
OUTPUT_FILE = "weather_final.csv"



In [23]:

# ==========================================
# 2. PRÉPARATION DES VILLES
# ==========================================
print(" Chargement des coordonnées...")
try:
    df_hotels = pd.read_csv(INPUT_FILE)
    df_cities = df_hotels.groupby('ville')[['latitude', 'longitude']].mean().reset_index()
    df_cities['city_id'] = df_cities.index + 1
    print(f" {len(df_cities)} villes à traiter.")
except FileNotFoundError:
    print(f" Fichier '{INPUT_FILE}' introuvable.")
    exit()


 Chargement des coordonnées...
 35 villes à traiter.


In [24]:

# ==========================================
# 3. RÉCUPÉRATION MÉTÉO (API GRATUITE 5 JOURS)
# ==========================================
print("\n Récupération des prévisions (API Standard)...")

weather_data = []



 Récupération des prévisions (API Standard)...


In [ ]:

for index, row in df_cities.iterrows():
    lat = row['latitude']
    lon = row['longitude']
    city = row['ville']
    
    # URL de l'API Standard (Forecast 5 days / 3 hours) #l'api 3.0 est désormais payante, je possedais déjà une clef en 2.5
    url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&units=metric&appid={API_KEY}"
    
    try:
        response = requests.get(url)
        data = response.json()
        
        if response.status_code == 200:
            # L'API renvoie 40 blocs de 3h (5 jours x 8 blocs)
            # On va agréger tout ça pour avoir une moyenne globale sur les 5 jours
            
            temp_sum = 0
            rain_sum = 0
            pop_sum = 0
            count = 0
            
            for item in data['list']:
                temp_sum += item['main']['temp']
                pop_sum += item.get('pop', 0)
                
                # La pluie est parfois dans 'rain' -> '3h'
                if 'rain' in item and '3h' in item['rain']:
                    rain_sum += item['rain']['3h']
                
                count += 1
            
            # Moyennes
            avg_temp = temp_sum / count
            total_rain = rain_sum # C'est un cumul
            avg_pop = (pop_sum / count) * 100
            
            # Score Météo (Arbitraire : Température - Pluie)
            score = avg_temp - (total_rain * 2)
            
            weather_data.append({
                'city_id': row['city_id'],
                'ville': city,
                'latitude': lat,
                'longitude': lon,
                'temperature_moyenne': round(avg_temp, 1),
                'pluie_totale_mm': round(total_rain, 1),
                'risque_pluie_pct': round(avg_pop, 0),
                'score_meteo': round(score, 1)
            })
            print(f"    {city} : {round(avg_temp, 1)}°C, Pluie: {round(total_rain, 1)}mm")
            
        else:
            print(f"    Erreur {response.status_code} pour {city}: {data.get('message')}")
            
    except Exception as e:
        print(f"    Crash sur {city} : {e}")
        
    time.sleep(0.2) # Petite pause


    Aigues Mortes : 10.8°C, Pluie: 0.8mm
    Aix en Provence : 8.1°C, Pluie: 13.5mm
    Amiens : 8.3°C, Pluie: 13.2mm
    Annecy : 5.7°C, Pluie: 5.0mm
    Ariege : 7.2°C, Pluie: 12.7mm
    Avignon : 8.1°C, Pluie: 10.3mm
    Bayeux : 9.1°C, Pluie: 19.0mm
    Bayonne : 11.7°C, Pluie: 87.6mm
    Besancon : 6.9°C, Pluie: 6.0mm
    Biarritz : 12.2°C, Pluie: 85.3mm
    Bormes les Mimosas : 8.8°C, Pluie: 30.8mm
    Carcassonne : 10.3°C, Pluie: 13.3mm
    Cassis : 10.3°C, Pluie: 18.9mm
    Chateau du Haut Koenigsbourg : 4.6°C, Pluie: 10.5mm
    Collioure : 12.2°C, Pluie: 0.2mm
    Colmar : 7.1°C, Pluie: 5.8mm
    Dijon : 6.4°C, Pluie: 6.0mm
    Eguisheim : 6.8°C, Pluie: 5.6mm
    Gorges du Verdon : 3.5°C, Pluie: 28.6mm
    Grenoble : 7.1°C, Pluie: 10.3mm
    La Rochelle : 12.0°C, Pluie: 33.8mm
    Le Havre : 10.1°C, Pluie: 36.4mm
    Lille : 8.2°C, Pluie: 14.9mm
    Lyon : 7.3°C, Pluie: 1.6mm
    Marseille : 10.8°C, Pluie: 14.2mm
    Mont Saint Michel : 9.4°C, Pluie: 40.5mm
    Montauban : 9.6

In [26]:

# ==========================================
# 4. CLASSEMENT ET SAUVEGARDE
# ==========================================
if weather_data:
    df_weather = pd.DataFrame(weather_data)
    
    # Tri par "Meilleur temps" (Score le plus élevé)
    df_weather = df_weather.sort_values(by='score_meteo', ascending=False)
    
    # Sauvegarde
    df_weather.to_csv(OUTPUT_FILE, index=False)
    
    
    print("\n" + "="*40)
    print(" TOP 5 DESTINATIONS (MÉTÉO)")
    print("="*40)
    print(df_weather[['ville', 'temperature_moyenne', 'pluie_totale_mm']].head(5).to_string(index=False))
    
    
    print(f"\n Fichier '{OUTPUT_FILE}' généré avec succès.")
else:
    print("\n Aucune donnée récupérée.")


 TOP 5 DESTINATIONS (MÉTÉO)
                   ville  temperature_moyenne  pluie_totale_mm
               Collioure                 12.2              0.2
           Aigues Mortes                 10.8              0.8
                   Nimes                  8.5              0.5
Saintes Maries de la mer                 10.8              1.8
                    Uzes                  7.8              0.6

 Fichier 'weather_final.csv' généré avec succès.
